In [1]:
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from torch.utils.data import DataLoader, TensorDataset, random_split
from dgl.data import TUDataset

# Fixed I/O based on the N_MAX "Compile-time" boundary
N_MAX = 28 
INPUT_DIM = N_MAX * N_MAX  # 784
OUTPUT_BASIS = 15          
OUTPUT_DIM = OUTPUT_BASIS * N_MAX * N_MAX # 11,760

print(f"Parrot configuration: Input={INPUT_DIM}, Output={OUTPUT_DIM}")

Parrot configuration: Input=784, Output=11760


In [3]:
def ops_2_to_2(inputs, dim, normalization='inf'): 
    # inputs: N x D x m x m
    diag_part = th.diagonal(inputs, dim1=2, dim2=3) 
    sum_diag_part = th.sum(diag_part, dim=2, keepdim=True) 
    sum_of_rows = th.sum(inputs, dim=3) 
    sum_of_cols = th.sum(inputs, dim=2) 
    sum_all = th.sum(sum_of_rows, dim=2) 

    # Generate the 15 operators (Basis)
    op1 = th.diag_embed(diag_part)
    op2 = th.diag_embed(sum_diag_part.repeat(1, 1, dim))
    op3 = th.diag_embed(sum_of_rows)
    op4 = th.diag_embed(sum_of_cols)
    op5 = th.diag_embed(sum_all.unsqueeze(2).repeat(1, 1, dim))
    op6 = sum_of_cols.unsqueeze(3).repeat(1, 1, 1, dim)
    op7 = sum_of_rows.unsqueeze(3).repeat(1, 1, 1, dim)
    op8 = sum_of_cols.unsqueeze(2).repeat(1, 1, dim, 1)
    op9 = sum_of_rows.unsqueeze(2).repeat(1, 1, dim, 1)
    op10 = inputs
    op11 = th.transpose(inputs, -2, -1)
    op12 = diag_part.unsqueeze(3).repeat(1, 1, 1, dim)
    op13 = diag_part.unsqueeze(2).repeat(1, 1, dim, 1)
    op14 = sum_diag_part.unsqueeze(3).repeat(1, 1, dim, dim)
    op15 = sum_all.unsqueeze(2).unsqueeze(3).repeat(1, 1, dim, dim)

    if normalization == 'inf':
        float_dim = float(dim)
        op2, op3, op4 = op2/float_dim, op3/float_dim, op4/float_dim
        op5 = op5 / (float_dim**2)
        op6, op7, op8, op9 = op6/float_dim, op7/float_dim, op8/float_dim, op9/float_dim
        op14 = op14 / float_dim
        op15 = op15 / (float_dim**2)

    return [op1, op2, op3, op4, op5, op6, op7, op8, op9, op10, op11, op12, op13, op14, op15]

def pad_adjacency_matrix(adj, n_max=N_MAX):
    n = adj.shape[0]
    padded = th.zeros((n_max, n_max))
    padded[:n, :n] = adj
    return padded.unsqueeze(0).unsqueeze(0)

In [4]:
def collect_parrot_data(dataset, num_permutations=200):
    inputs_list = []
    outputs_list = []
    
    # Paper logic: The function is "Pure". 
    # Input: Adjacency Matrix -> Output: 15 Basis Operators
    for graph, label in dataset:
        adj = graph.adjacency_matrix().to_dense()
        num_nodes = adj.shape[0]
        
        # Canonical Target
        canonical_input = th.zeros((1, 1, N_MAX, N_MAX))
        canonical_input[0, 0, :num_nodes, :num_nodes] = adj
        
        with th.no_grad():
            # ops_2_to_2 is the "Original C++ Function" we are replacing
            basis_list = ops_2_to_2(canonical_input, N_MAX)
            canonical_output = th.stack(basis_list, dim=0).squeeze() 
        
        # Data Augmentation: Teaching the MLP that different shuffles = same output
        for _ in range(num_permutations):
            perm = th.randperm(num_nodes)
            shuffled_adj = adj[perm][:, perm]
            
            padded_input = th.zeros((N_MAX, N_MAX))
            padded_input[:num_nodes, :num_nodes] = shuffled_adj
            
            inputs_list.append(padded_input.flatten()) 
            outputs_list.append(canonical_output.view(-1))         
            
    return th.stack(inputs_list), th.stack(outputs_list)

dataset = TUDataset('MUTAG')
raw_inputs, raw_outputs = collect_parrot_data(dataset, num_permutations=200)

# Normalization (as required by the paper's blueprint)
X_min, X_max = raw_inputs.min(), raw_inputs.max()
y_min, y_max = raw_outputs.min(), raw_outputs.max()

X_norm = (raw_inputs - X_min) / (X_max - X_min + 1e-7)
y_norm = (raw_outputs - y_min) / (y_max - y_min + 1e-7)

full_dataset = TensorDataset(X_norm, y_norm)
# 70/30 split as per paper
train_size = int(0.7 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [5]:
class ParrotMLP(nn.Module):
    def __init__(self, layers):
        super(ParrotMLP, self).__init__()
        layer_list = []
        for i in range(len(layers) - 1):
            layer_list.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                # The paper specifies Sigmoid for internal neurons
                layer_list.append(nn.Sigmoid())
        self.model = nn.Sequential(*layer_list)

    def forward(self, x):
        return self.model(x)

def get_paper_topologies():
    """Generates the search space restricted by the paper."""
    topologies = []
    # Powers of 2 for hidden layers
    neuron_options = [16, 32, 64, 128, 256, 512] 
    
    # 1 hidden layer
    for n in neuron_options:
        topologies.append([INPUT_DIM, n, OUTPUT_DIM])
        
    # 2 hidden layers
    for n1 in neuron_options:
        for n2 in neuron_options:
            if n1 >= n2: # Typical heuristic to reduce search
                topologies.append([INPUT_DIM, n1, n2, OUTPUT_DIM])
    return topologies[:30] # Limit to 30 variations as per paper context

In [6]:
def train_and_evaluate(topology, epochs=50):
    model = ParrotMLP(topology).cuda() if th.cuda.is_available() else ParrotMLP(topology)
    optimizer = th.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # Training
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            if th.cuda.is_available(): batch_X, batch_y = batch_X.cuda(), batch_y.cuda()
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation on 30% Test Set
    model.eval()
    total_mse = 0
    with th.no_grad():
        for batch_X, batch_y in test_loader:
            if th.cuda.is_available(): batch_X, batch_y = batch_X.cuda(), batch_y.cuda()
            total_mse += criterion(model(batch_X), batch_y).item()
    
    avg_mse = total_mse / len(test_loader)
    return avg_mse, model.cpu()

# Executing the search
all_topologies = get_paper_topologies()
results = []
for top in all_topologies[::5]: # Sampling for speed, or run all for full proof
    print(f"Testing Topology: {top}")
    mse, trained_model = train_and_evaluate(top)
    results.append({'topology': top, 'mse': mse, 'model': trained_model})

winner = min(results, key=lambda x: x['mse'])
print(f"\nWINNER TOPOLOGY: {winner['topology']} with MSE: {winner['mse']:.6f}")

Testing Topology: [784, 16, 11760]
Testing Topology: [784, 512, 11760]
Testing Topology: [784, 64, 32, 11760]
Testing Topology: [784, 128, 128, 11760]
Testing Topology: [784, 256, 256, 11760]
Testing Topology: [784, 512, 256, 11760]

WINNER TOPOLOGY: [784, 64, 32, 11760] with MSE: 0.001881


In [7]:
def test_invariance_failure(graph_idx=0):
    graph, _ = dataset[graph_idx]
    adj = graph.adjacency_matrix().to_dense()
    num_nodes = adj.shape[0]
    
    # Original vs Shuffled
    perm = th.randperm(num_nodes)
    shuffled_adj = adj[perm][:, perm]
    
    # Prep inputs
    in_orig = (pad_adjacency_matrix(adj).flatten() - X_min) / (X_max - X_min + 1e-7)
    in_shuff = (pad_adjacency_matrix(shuffled_adj).flatten() - X_min) / (X_max - X_min + 1e-7)
    
    winner_model = winner['model']
    winner_model.eval()
    
    with th.no_grad():
        out_orig = winner_model(in_orig.unsqueeze(0))
        out_shuff = winner_model(in_shuff.unsqueeze(0))
    
    inv_error = th.abs(out_orig - out_shuff).mean().item()
    
    print(f"--- Generalization Report ---")
    print(f"Model Topology: {winner['topology']}")
    print(f"Invariance Error (Same graph, different order): {inv_error:.8f}")
    
    if inv_error > 0.01:
        print("CONCLUSION: The MLP is approximating values, but failing the 'Pure Function' logic.")
        print("It has not learned the structural rules of Graph Isomorphism.")

test_invariance_failure(0)

--- Generalization Report ---
Model Topology: [784, 64, 32, 11760]
Invariance Error (Same graph, different order): 0.00244877
